# Phase 2 — Profile Research

**Exit gate (plan §9):** a written finding — which channel combinations
actually produce high points-per-90 by position, with the DEFCON
threshold-rate analysis. The hypothesis may be falsified here; that's a
valid outcome (plan principle 6: "Kill findings that don't hold").

Data: 3 completed seasons pulled via `fpl.collect.history_loader` from the
vaastav/Fantasy-Premier-League archive — `2023-24`, `2024-25`, `2025-26`.
2025-26 is the only one with DEFCON scoring (plan §3.2), so the DEFCON
analysis in this notebook uses that season alone; the other two feed the
goals/assists/clean-sheet/bonus baselines.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import yaml

ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent  # notebook is usually opened with cwd = notebooks/
HIST_DIR = ROOT / "data" / "raw" / "history"

with open(ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

SEASONS = config["history"]["seasons"]
GOAL_MULT = config["position_multipliers"]["goals"]
ASSIST_MULT = config["position_multipliers"]["assists_flat"]
CS_VALUE = config["position_multipliers"]["clean_sheet_value"]
DEFCON_THRESH = {
    "DEF": config["defcon"]["def_threshold_cbit"],
    "MID": config["defcon"]["mid_fwd_threshold_cbirt"],
    "FWD": config["defcon"]["mid_fwd_threshold_cbirt"],
}
DEFCON_PTS = config["defcon"]["points"]

frames = []
for season in SEASONS:
    path = HIST_DIR / season / "gws" / "merged_gw.csv"
    df = pd.read_csv(path, encoding="utf-8")
    df["season"] = season
    frames.append(df)
    print(f"{season}: {len(df)} player-gameweek rows")
all_gw = pd.concat(frames, ignore_index=True)
print(f"\nTotal player-gameweek rows across {len(SEASONS)} seasons: {len(all_gw)}")

2023-24: 29725 player-gameweek rows
2024-25: 27605 player-gameweek rows
2025-26: 29757 player-gameweek rows

Total player-gameweek rows across 3 seasons: 87087


## Section 1 — Player-season aggregation

Roll each season's per-gameweek rows up to one row per player per season, and
qualify on minutes played (>= 450, i.e. ~5 full matches) so a couple of cameo
appearances don't distort a per-90 rate.

In [2]:
MIN_MINUTES = 450

agg_cols = {
    "minutes": "sum", "goals_scored": "sum", "assists": "sum",
    "clean_sheets": "sum", "bonus": "sum", "total_points": "sum",
}
player_season = all_gw.groupby(["season", "element", "name", "position"], as_index=False).agg(agg_cols)

qualified = player_season[player_season["minutes"] >= MIN_MINUTES].copy()
print(f"Player-seasons total: {len(player_season)}, qualified (>= {MIN_MINUTES} min): {len(qualified)}")
print(qualified["position"].value_counts())

Player-seasons total: 2515, qualified (>= 450 min): 1200
position
MID    538
DEF    444
FWD    125
GK      93
Name: count, dtype: int64


## Section 2 — Per-90 channel decomposition, by position

Convert each channel's per-90 rate into points, using the position
multipliers from `config.yaml` (goal points by position, flat 3 for assists,
clean-sheet value by position) — the same constants `fpl/project/baseline.py`
will use in Phase 3.

In [3]:
for col in ["goals_scored", "assists", "clean_sheets", "bonus", "total_points"]:
    qualified[f"{col}_per90"] = qualified[col] * 90 / qualified["minutes"]

qualified["goal_pts_per90"] = qualified.apply(lambda r: r["goals_scored_per90"] * GOAL_MULT.get(r["position"], 0), axis=1)
qualified["assist_pts_per90"] = qualified["assists_per90"] * ASSIST_MULT
qualified["cs_pts_per90"] = qualified.apply(lambda r: r["clean_sheets_per90"] * CS_VALUE.get(r["position"], 0), axis=1)
qualified["bonus_pts_per90"] = qualified["bonus_per90"]

summary_cols = ["goal_pts_per90", "assist_pts_per90", "cs_pts_per90", "bonus_pts_per90", "total_points_per90"]
print("Mean per-90 channel points by position (3-season pool, DEFCON excluded — see Section 4):\n")
print(qualified.groupby("position")[summary_cols].mean().round(2).to_string())

Mean per-90 channel points by position (3-season pool, DEFCON excluded — see Section 4):

          goal_pts_per90  assist_pts_per90  cs_pts_per90  bonus_pts_per90  total_points_per90
position                                                                                     
DEF                 0.26              0.21          0.90             0.19                3.23
FWD                 1.62              0.46          0.00             0.58                5.08
GK                  0.00              0.02          0.85             0.21                3.21
MID                 0.79              0.52          0.26             0.27                4.22


**Reading this table:** these four channels don't sum to `total_points_per90`
— appearance points, cards, saves/conceded (GK/DEF), and (for 2025-26)
DEFCON all sit outside this decomposition. What it does show cleanly: goals
dominate FWD scoring, clean sheets dominate DEF/GK scoring, and MID sits
between the two — which is the shape the plan's channel weighting assumes.

## Section 3 — Which channels actually drive points-per-90?

Correlate each channel's per-90 points with total points-per-90, within
position — this is the plan's actual question ("which channel combinations
actually produce high points-per-90 by position"), not just which channel is
biggest on average.

In [4]:
channel_cols = ["goal_pts_per90", "assist_pts_per90", "cs_pts_per90", "bonus_pts_per90"]
for pos in ["GK", "DEF", "MID", "FWD"]:
    sub = qualified[qualified["position"] == pos]
    corrs = sub[channel_cols].corrwith(sub["total_points_per90"]).dropna().sort_values(ascending=False)
    print(f"{pos} (n={len(sub)}):")
    print(corrs.round(3).to_string())
    print()

GK (n=93):
cs_pts_per90        0.777
bonus_pts_per90     0.601
assist_pts_per90    0.227

DEF (n=444):
cs_pts_per90        0.775
bonus_pts_per90     0.661
goal_pts_per90      0.465
assist_pts_per90    0.432

MID (n=538):
goal_pts_per90      0.822
bonus_pts_per90     0.742
assist_pts_per90    0.652
cs_pts_per90        0.287

FWD (n=125):
goal_pts_per90      0.861
bonus_pts_per90     0.757
assist_pts_per90    0.263



**Finding:** clean sheets are the strongest single driver for GK and DEF
(corr ~0.78 both); goals are the strongest driver for MID and FWD (~0.76-0.86).
Bonus points track total points closely everywhere (~0.6-0.76) — expected,
since bonus is itself partly derived from a strong match. Assists matter more
for MID than for FWD or DEF. Nothing here overturns the plan's channel
weighting — it confirms the intuitive shape and gives it numbers.

## Section 4 — DEFCON threshold-rate analysis (2025-26 only)

Plan §4.2's claim: the binary threshold means a player's *rate of crossing
the line* is the right estimator, not their average action count — "a player
averaging 11 CBIT is not '1.1x a 10-threshold player'". This section checks
that claim against real 2025-26 data, using `defensive_contribution`
directly — confirmed below to already be the position-correct raw count
(CBIT for DEF, CBIRT for MID/FWD), matching FPL's own scoring rule exactly.

In [5]:
gw2526 = all_gw[all_gw["season"] == "2025-26"].copy()

# Confirm defensive_contribution's formula is position-correct before relying on it.
sample_def = gw2526[(gw2526["position"] == "DEF") & (gw2526["minutes"] > 0)].head(200)
def_match = (sample_def["defensive_contribution"] == sample_def["clearances_blocks_interceptions"] + sample_def["tackles"]).mean()
sample_mid = gw2526[(gw2526["position"] == "MID") & (gw2526["minutes"] > 0)].head(200)
mid_match = (sample_mid["defensive_contribution"] == sample_mid["clearances_blocks_interceptions"] + sample_mid["tackles"] + sample_mid["recoveries"]).mean()
print(f"DEF rows where defensive_contribution == CBI+tackles (CBIT):        {def_match:.1%}")
print(f"MID rows where defensive_contribution == CBI+tackles+recoveries (CBIRT): {mid_match:.1%}")

DEF rows where defensive_contribution == CBI+tackles (CBIT):        100.0%
MID rows where defensive_contribution == CBI+tackles+recoveries (CBIRT): 100.0%


In [6]:
played = gw2526[gw2526["minutes"] > 0].copy()
played["threshold"] = played["position"].map(DEFCON_THRESH)
played = played.dropna(subset=["threshold"])  # GK has no DEFCON
played["over_threshold"] = played["defensive_contribution"] >= played["threshold"]

player_defcon = played.groupby(["element", "name", "position"], as_index=False).agg(
    matches_played=("minutes", "size"),
    matches_over=("over_threshold", "sum"),
    mean_dc=("defensive_contribution", "mean"),
)
player_defcon["rate"] = player_defcon["matches_over"] / player_defcon["matches_played"]
player_defcon["threshold"] = player_defcon["position"].map(DEFCON_THRESH)
player_defcon["avg_proxy"] = (player_defcon["mean_dc"] / player_defcon["threshold"]).clip(upper=1.0)

MIN_MATCHES = 10
pdq = player_defcon[player_defcon["matches_played"] >= MIN_MATCHES].copy()
print(f"Qualified players (>= {MIN_MATCHES} matches played, DEF/MID/FWD): {len(pdq)}\n")

for pos in ["DEF", "MID", "FWD"]:
    sub = pdq[pdq["position"] == pos].sort_values("rate", ascending=False)
    print(f"Top 8 {pos} by DEFCON threshold-crossing rate:")
    print(sub[["name", "matches_played", "matches_over", "rate", "mean_dc"]].head(8).round(2).to_string(index=False))
    print()

Qualified players (>= 10 matches played, DEF/MID/FWD): 389

Top 8 DEF by DEFCON threshold-crossing rate:
                   name  matches_played  matches_over  rate  mean_dc
    Marcos Senesi Barón              37            26  0.70    11.32
       Joachim Andersen              33            20  0.61     9.97
        James Tarkowski              37            22  0.59    10.16
Konstantinos Mavropanos              31            18  0.58    10.10
        Maxence Lacroix              35            20  0.57    10.57
          Maxime Estève              34            19  0.56     9.82
         Daniel Ballard              29            15  0.52     9.10
            Kevin Danso              24            12  0.50     8.00

Top 8 MID by DEFCON threshold-crossing rate:
                         name  matches_played  matches_over  rate  mean_dc
              Elliot Anderson              38            26  0.68    13.55
                 Ethan Ampadu              35            19  0.54    11.89
   

**Sense check:** the top DEF names (Senesi, Andersen, Tarkowski, Mavropanos)
are all primarily-defensive centre-backs; the top MID names (Elliot
Anderson, Ampadu, Garner, Bentancur) are all holding/defensive
midfielders. Forwards barely cross the threshold at all — Bowen, a
wide forward who tracks back, tops that list at a 16% rate versus DEF/MID
leaders above 50-70%. This matches football reality and is a reasonable
sanity check that the metric isn't picking up noise.

In [7]:
diff = (pdq["rate"] - pdq["avg_proxy"]).abs()
corr = pdq["rate"].corr(pdq["avg_proxy"])
print(f"Mean |rate - avg_proxy| across {len(pdq)} qualified players: {diff.mean():.3f}")
print(f"Correlation(rate, avg_proxy): {corr:.3f}\n")

biggest = pdq.assign(diff=diff).sort_values("diff", ascending=False).head(8)
print("Biggest divergences between the rate estimator and the naive average-based proxy:")
print(biggest[["name", "position", "matches_played", "mean_dc", "rate", "avg_proxy", "diff"]].round(2).to_string(index=False))

Mean |rate - avg_proxy| across 389 qualified players: 0.357
Correlation(rate, avg_proxy): 0.890

Biggest divergences between the rate estimator and the naive average-based proxy:
                  name position  matches_played  mean_dc  rate  avg_proxy  diff
              Ola Aina      DEF              18     7.22  0.11       0.72  0.61
          Matt Doherty      DEF              11     6.00  0.00       0.60  0.60
       Youri Tielemans      MID              25     8.12  0.08       0.68  0.60
       Ladislav Krejcí      DEF              28     7.04  0.11       0.70  0.60
   Quilindschy Hartman      DEF              21     7.29  0.14       0.73  0.59
        Mohammed Kudus      MID              19     7.00  0.00       0.58  0.58
         Omar Alderete      DEF              33     9.45  0.36       0.95  0.58
Santiago Ignacio Bueno      DEF              29     7.45  0.17       0.74  0.57


**Finding: plan §4.2's claim holds.** The average-based proxy systematically
*overestimates* the true crossing rate — every example above has
`avg_proxy > rate`, several by 0.5-0.6 (i.e. the naive method says "usually
clears the threshold," the actual rate says "rarely does"). A player who
averages just under threshold every match (Doherty: mean_dc=6.0, avg_proxy
implies 0.60 but actual rate=0.0 across 11 matches) scores nothing on
DEFCON despite a respectable average, because 6 CBIT never crosses a 10-CBIT
line. The rate-based estimator (`matches_over_threshold / matches_played`,
exactly as plan §4.2 specifies) is the only one of the two that isn't
misleading here — confirmed, not just asserted.

## Section 5 — Does DEFCON predict who scores well overall?

Fold DEFCON points into the 2025-26 per-90 channel/correlation picture from
Sections 2-3, now that this season actually has the data.

In [8]:
played["defcon_points"] = np.where(played["over_threshold"], DEFCON_PTS, 0)

season_2526 = played.groupby(["element", "name", "position"], as_index=False).agg(
    minutes=("minutes", "sum"), defcon_points=("defcon_points", "sum"),
)
# bring in the non-defensive channels for 2025-26 from the earlier `qualified` table
q2526 = qualified[qualified["season"] == "2025-26"].merge(
    season_2526[["element", "defcon_points"]], on="element", how="left"
)
q2526["defcon_points"] = q2526["defcon_points"].fillna(0)
q2526["defcon_pts_per90"] = q2526["defcon_points"] * 90 / q2526["minutes"]

cols = ["goal_pts_per90", "assist_pts_per90", "cs_pts_per90", "bonus_pts_per90", "defcon_pts_per90", "total_points_per90"]
print("2025-26 only, DEF/MID — mean per-90 channel points (DEFCON included):\n")
print(q2526[q2526["position"].isin(["DEF", "MID"])].groupby("position")[cols].mean().round(2).to_string())

print("\nCorrelation with total_points_per90 (DEFCON included):\n")
for pos in ["DEF", "MID"]:
    sub = q2526[q2526["position"] == pos]
    corrs = sub[cols[:-1]].corrwith(sub["total_points_per90"]).round(3).sort_values(ascending=False)
    print(f"{pos} (n={len(sub)}):")
    print(corrs.to_string())
    print()

print("Share of mean total_points_per90 contributed by each channel, DEF:")
sub = q2526[q2526["position"] == "DEF"]
for c in cols[:-1]:
    print(f"  {c}: {sub[c].mean() / sub['total_points_per90'].mean() * 100:.1f}%")

2025-26 only, DEF/MID — mean per-90 channel points (DEFCON included):

          goal_pts_per90  assist_pts_per90  cs_pts_per90  bonus_pts_per90  defcon_pts_per90  total_points_per90
position                                                                                                       
DEF                 0.24              0.22          1.00             0.20              0.49                3.77
MID                 0.70              0.51          0.27             0.28              0.29                4.36

Correlation with total_points_per90 (DEFCON included):

DEF (n=147):
cs_pts_per90        0.778
bonus_pts_per90     0.700
goal_pts_per90      0.442
assist_pts_per90    0.421
defcon_pts_per90    0.069

MID (n=181):
goal_pts_per90      0.762
bonus_pts_per90     0.709
assist_pts_per90    0.557
cs_pts_per90        0.300
defcon_pts_per90   -0.165

Share of mean total_points_per90 contributed by each channel, DEF:
  goal_pts_per90: 6.3%
  assist_pts_per90: 5.9%
  cs_pts_per90: 26.5%

**This is the finding worth flagging.** DEFCON contributes a real, non-trivial
share of DEF points-per-90 (~13%, comparable to bonus and roughly half of
clean sheets' share) — it clearly matters and belongs in the model as an
additive term, exactly as plan §4.1 structures it. But its **correlation
with total points-per-90 is weak for DEF (~0.07) and slightly negative for
MID**, unlike every other channel, which all correlate positively (0.3-0.86).

**Why, and what it means for the model:** high-DEFCON players tend to be
defence-first players on teams that create/concede at a level that caps
their goals/assists/bonus upside — so DEFCON output doesn't track "this is a
good FPL player" the way goals or clean sheets do. It's not redundant with
the other channels, and it's not a proxy for overall quality either — it's
a genuinely independent scoring axis. That's exactly why the plan treats
`defcon_pts` as its own rate-estimated term rather than folding it into a
general per-90 "form" signal — this data confirms that structural choice
was right, not just convenient. It also means the optimiser will find real,
separate value in DEFCON-specialist defenders/mids that a model without
this term would miss entirely.

## Section 6 — Summary of findings

1. **Channel drivers by position** (Section 3): clean sheets drive GK/DEF
   points-per-90 (corr ~0.78 both); goals drive MID/FWD (~0.76-0.86); bonus
   tracks total points everywhere; assists matter most for MID. Confirms
   the plan's channel weighting rather than overturning it.
2. **DEFCON threshold-rate estimator is correct as specified** (Section 4):
   the naive average-based proxy overestimates crossing rate by 0.36 on
   average across qualified players, several by 0.5+. The plan's
   `matches_over_threshold / matches_played` estimator is the one that
   isn't misleading — verified against real 2025-26 data, not assumed.
3. **DEFCON is a non-redundant, independent scoring channel** (Section 5):
   it contributes ~13% of DEF points-per-90 but correlates weakly-to-
   negatively with total points-per-90 — it doesn't track "good player,"
   it tracks a genuinely separate skill. Reinforces modelling it as its
   own additive term with its own per-player rate estimate.
4. **Top DEFCON performers pass the football eye test** (Section 4):
   centre-backs and holding midfielders dominate both lists; forwards
   barely register. No sign the metric is picking up noise.

**A data issue found and fixed during this phase, not a modelling
finding but worth recording:** `config.yaml`'s `season`/`history.seasons`
were initially set assuming "2025-26" was the current season, per the
plan's literal wording. Cross-checking against today's date and the
archive (2025-26 has full DEFCON columns for all 38 GWs; 2026-27 404s —
not created yet) showed 2026-27 is actually the upcoming season the plan's
"~48h to GW1" refers to. Fixed before this analysis ran; see the
`fix: correct season config` commit.

**Limitations carried into Phase 3:** the 450-minute qualification floor
and 10-match floor for the DEFCON tables are judgment calls, not derived —
worth sensitivity-checking once the backtest (Phase 3 exit gate) is running.
This notebook doesn't yet weight by fixture difficulty or split home/away,
both of which plan §4.2 requires in the actual projection model.